# 04 - Comparisons

Compares finished runs from `experiments/` using the metrics each run saved, so no model is retrained here.

1. ConvVAE: k-means vs GMM.
2. Temporal VAE: k-means vs GMM.
3. ConvVAE vs Temporal VAE, using the winner of each comparison above.

Metrics are grouped in two families:
- Separation: silhouette (higher is better), Davies-Bouldin (lower is better) and Calinski-Harabasz (higher is better).
- Persistence: mean run length (higher is better) and switches per year (lower is better).

A run wins a family when it wins more of its metrics. The overall winner wins both families, or one family with the other tied. If each run wins one family, the split is resolved by total metric wins and marked as split.

Results are saved to `experiments/comparisons.csv` (one row per pair and metric) and `experiments/comparisons_summary.csv` (one row per pair). Comparing a pair again replaces its rows.

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import pandas as pd

from src.comparison import compare, report, summary_table, save_comparisons

## 0. Runs to compare

Write the run ids (folder paths inside `experiments/`) of the k-means and GMM run of each model. Both runs of a pair should use the same scalogram set and k. Set `MODEL_PAIR` only to override the automatic choice in section 3.

In [ ]:
ALGORITHM_PAIRS = {
    "conv_vae": ("conv_vae/001_real_price_kmeans_k2", "conv_vae/002_real_price_gmm_k2"),
    "temporal_vae": ("temporal_vae/001_real_price_kmeans_k2", "temporal_vae/002_real_price_gmm_k2"),
}
MODEL_PAIR = None

DATA_DIR = Path("../data/processed")
EXPERIMENTS_DIR = Path("../experiments")

## 1. ConvVAE: k-means vs GMM

In [ ]:
conv = compare(*ALGORITHM_PAIRS["conv_vae"], DATA_DIR, EXPERIMENTS_DIR)
report(conv)

## 2. Temporal VAE: k-means vs GMM

In [ ]:
temporal = compare(*ALGORITHM_PAIRS["temporal_vae"], DATA_DIR, EXPERIMENTS_DIR)
report(temporal)

## 3. ConvVAE vs Temporal VAE

Uses the overall winner of sections 1 and 2. If one of them is a tie, set `MODEL_PAIR` in section 0.

In [ ]:
if MODEL_PAIR is None:
    winners = (conv["summary"]["overall_winner"], temporal["summary"]["overall_winner"])
    if "tie" in winners:
        raise ValueError("A k-means vs GMM comparison is tied. Set MODEL_PAIR in section 0.")
    MODEL_PAIR = winners

models = compare(*MODEL_PAIR, DATA_DIR, EXPERIMENTS_DIR)
report(models)

## 4. Summary

In [ ]:
results = [conv, temporal, models]
print(summary_table([r["summary"] for r in results]))
for r in results:
    print(f"- {r['summary']['verdict']}")

In [ ]:
summary = save_comparisons(results, EXPERIMENTS_DIR)
summary[["date", "run_a", "run_b", "overall_winner", "basis", "changed", "same_regime_share"]]